In [1]:
# ============================================================================
# Kaggle-Optimized: Bottleneck Formation in Deep Networks Under Joint Scarcity
# Version 5 - Final with NaN/Inf handling and all 11 plots
# ============================================================================

# %% Cell 1: Imports and Setup
import os
import sys
import pickle
import hashlib
import time
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd as scipy_svd
import pandas as pd

print("✓ Imports successful!")

# %% Cell 2: Kaggle Environment Setup
def setup_kaggle_environment():
    """Setup paths and check Kaggle environment."""
    if os.path.exists('/kaggle/working'):
        BASE_DIR = Path('/kaggle/working')
    else:
        BASE_DIR = Path('.')
    
    RESULTS_DIR = BASE_DIR / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
    
    PLOTS_DIR = BASE_DIR / 'plots'
    PLOTS_DIR.mkdir(exist_ok=True)
    
    CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    print(f"✓ Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"✓ Results directory: {RESULTS_DIR}")
    print(f"✓ Plots directory: {PLOTS_DIR}")
    print(f"✓ PyTorch: {torch.__version__}")
    print(f"✓ Lightning: {pl.__version__}")
    
    return RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE

RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE = setup_kaggle_environment()

# %% Cell 3: Time and Memory Management
class ExperimentTimer:
    """Manage experiment time to stay within Kaggle limits."""
    
    def __init__(self, max_hours: float = 11.5):
        self.start_time = time.time()
        self.max_seconds = max_hours * 3600
        
    def elapsed_hours(self) -> float:
        return (time.time() - self.start_time) / 3600
    
    def remaining_hours(self) -> float:
        return self.max_seconds / 3600 - self.elapsed_hours()
    
    def should_stop(self) -> bool:
        return self.elapsed_hours() >= (self.max_seconds / 3600)
    
    def print_status(self):
        print(f"⏱️  Elapsed: {self.elapsed_hours():.2f}h | Remaining: {self.remaining_hours():.2f}h")

def cleanup_memory():
    """Aggressive memory cleanup."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_memory_stats():
    """Get current memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        return f"GPU: {allocated:.2f}GB allocated, {cached:.2f}GB cached"
    return "CPU mode"

# %% Cell 4: Reproducibility
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    pl.seed_everything(seed, workers=True)

# %% Cell 5: Dataset Generation
class GaussianBlobsDataset(Dataset):
    """Synthetic dataset: K classes on hypersphere with Gaussian noise."""
    
    def __init__(self, K: int, d: int, n_per_class: int, seed: int = 42):
        set_seed(seed)
        self.K = K
        self.d = d
        self.n_per_class = n_per_class
        self.N = K * n_per_class
        
        centers = self._generate_centers()
        
        X_list, y_list = [], []
        for k in range(K):
            X_k = np.random.randn(n_per_class, d) + centers[k]
            y_k = np.full(n_per_class, k)
            X_list.append(X_k)
            y_list.append(y_k)
        
        self.X = torch.FloatTensor(np.vstack(X_list))
        self.y = torch.LongTensor(np.hstack(y_list))
        
    def _generate_centers(self) -> np.ndarray:
        """Generate K orthogonal class centers on sqrt(d) * hypersphere."""
        A = np.random.randn(self.d, self.K)
        Q, _ = np.linalg.qr(A)
        centers = np.sqrt(self.d) * Q.T[:self.K]
        return centers
    
    def __len__(self):
        return self.N
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_data_loaders(K: int, d: int, n_per_class: int, 
                        batch_size: int, seed: int = 42):
    """Create train/val loaders with 90/10 split."""
    dataset = GaussianBlobsDataset(K, d, n_per_class, seed)
    N = len(dataset)
    n_val = max(1, int(0.1 * N))
    n_train = N - n_val
    
    train_ds, val_ds = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(seed)
    )
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                              num_workers=0, persistent_workers=False, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                           num_workers=0, persistent_workers=False, pin_memory=False)
    
    return train_loader, val_loader, dataset

# %% Cell 6: Network Architecture (UPDATE)
class BottleneckMLP(nn.Module):
    """Deep MLP with configurable bottleneck architecture."""
    
    def __init__(self, d0: int, K: int, L_hidden: int, d_b: int,
                 width_pattern: str = 'constant_narrow', use_relu: bool = True,
                 init_type: str = 'kaiming'): # CONTROL EXPERIMENT ADDITION: Added init_type
        super().__init__()
        self.d0 = d0
        self.K = K
        self.L_hidden = L_hidden
        self.d_b = d_b
        self.width_pattern = width_pattern
        self.use_relu = use_relu
        
        widths = self._build_widths()
        self.widths = widths
        
        layers = []
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i+1]))
            if i < len(widths) - 2:
                if use_relu:
                    layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
        
        # CONTROL EXPERIMENT ADDITION: Apply initialization
        self._apply_initialization(init_type)

    def _apply_initialization(self, init_type: str):
        """Apply specific initialization scheme."""
        def init_weights(m):
            if isinstance(m, nn.Linear):
                if init_type == 'xavier':
                    nn.init.xavier_uniform_(m.weight)
                elif init_type == 'kaiming':
                    nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        
        self.network.apply(init_weights)
    
    def _build_widths(self) -> List[int]:
        # ... (Existing implementation remains unchanged)
        if self.width_pattern == 'constant_narrow':
            widths = [self.d0] + [self.d_b] * self.L_hidden + [self.K]
        elif self.width_pattern == 'wide_narrow_wide':
            if self.L_hidden < 3:
                widths = [self.d0, self.d_b, self.K]
            else:
                n_wide = (self.L_hidden - 1) // 2
                widths = ([self.d0] + [256] * n_wide + [self.d_b] + 
                         [256] * (self.L_hidden - n_wide - 1) + [self.K])
        elif self.width_pattern == 'constant_wide':
            widths = [self.d0] + [1024] * self.L_hidden + [self.K]
        else:
            raise ValueError(f"Unknown width pattern: {self.width_pattern}")
        return widths
    
    def forward(self, x):
        return self.network(x)
    
    def get_layer_activations(self, x: torch.Tensor) -> List[torch.Tensor]:
        # ... (Existing implementation remains unchanged)
        activations = [x]
        h = x
        for layer in self.network:
            h = layer(h)
            if isinstance(layer, (nn.Linear, nn.ReLU)):
                activations.append(h.detach())
        return activations

# %% Cell 7: Metrics Computation (UPDATE)
class MetricsComputer:
    # ... (Existing stable_rank, nc1_metric, singular_values methods remain unchanged)
    @staticmethod
    def stable_rank(H: torch.Tensor) -> float:
        # (Existing code...)
        try:
            if H.shape[0] < 2: return 1.0
            frob_norm = torch.norm(H, p='fro').item() ** 2
            spec_norm = torch.linalg.matrix_norm(H, ord=2).item() ** 2
            if not np.isfinite(frob_norm) or not np.isfinite(spec_norm) or spec_norm < 1e-10:
                return 1.0
            rank = frob_norm / (spec_norm + 1e-10)
            if not np.isfinite(rank): return 1.0
            return max(1.0, min(rank, float(min(H.shape))))
        except Exception: return 1.0

    @staticmethod
    def nc1_metric(H: torch.Tensor, y: torch.Tensor, K: int) -> float:
        # (Existing code...)
        try:
            H_np = H.cpu().numpy()
            y_np = y.cpu().numpy()
            if not np.isfinite(H_np).all(): return 0.5
            mu_G = H_np.mean(axis=0)
            Sigma_W = 0; Sigma_T = 0
            for k in range(K):
                H_k = H_np[y_np == k]
                if len(H_k) == 0: continue
                mu_k = H_k.mean(axis=0)
                Sigma_W += (H_k - mu_k).T @ (H_k - mu_k)
                Sigma_T += len(H_k) * np.outer(mu_k - mu_G, mu_k - mu_G)
            Sigma_T += Sigma_W
            tr_W = np.trace(Sigma_W); tr_T = np.trace(Sigma_T)
            if not np.isfinite(tr_W) or not np.isfinite(tr_T) or tr_T < 1e-10: return 0.5
            return float(np.clip(tr_W / (tr_T + 1e-10), 0.0, 1.0))
        except Exception: return 0.5

    @staticmethod
    def singular_values(H: torch.Tensor, k: int = 10) -> np.ndarray:
        # (Existing code...)
        try:
            if H.shape[0] < k: k = H.shape[0]
            if not torch.isfinite(H).all(): return np.ones(k)
            _, S, _ = torch.svd_lowrank(H, q=min(k, min(H.shape)))
            return S.cpu().numpy()
        except Exception: return np.ones(k)

    # CONTROL EXPERIMENT ADDITION: Metric for dead ReLU / sparsity
    @staticmethod
    def compute_sparsity(H: torch.Tensor) -> float:
        """Compute fraction of zero activations (sparsity)."""
        try:
            return (H == 0).float().mean().item()
        except Exception:
            return 0.0

    @staticmethod
    def compute_all_metrics(model: nn.Module, loader: DataLoader, 
                           K: int, device: str) -> Dict:
        # ... (Existing setup code)
        model.eval()
        model = model.to(device)
        all_X, all_y = [], []
        for X, y in loader:
            all_X.append(X); all_y.append(y)
        X_full = torch.cat(all_X, dim=0).to(device)
        y_full = torch.cat(all_y, dim=0).to(device)
        
        with torch.no_grad():
            activations = model.get_layer_activations(X_full)
        
        metrics = {}
        L = len(activations) - 1
        
        for ell in range(1, L + 1):
            H = activations[ell]
            if len(H.shape) == 1: H = H.unsqueeze(0)
            if ell == L or H.shape[1] == K: continue
            
            srank = MetricsComputer.stable_rank(H)
            nc1 = MetricsComputer.nc1_metric(H, y_full, K)
            svals = MetricsComputer.singular_values(H, k=10)
            # CONTROL EXPERIMENT ADDITION: Compute sparsity
            sparsity = MetricsComputer.compute_sparsity(H)
            
            metrics[f'layer_{ell}'] = {
                'stable_rank': srank,
                'nc1': nc1,
                'singular_values': svals,
                'sparsity': sparsity, # Log sparsity
                'width': H.shape[1]
            }
        
        del X_full, y_full, activations
        cleanup_memory()
        return metrics

# %% Cell 8: Experiment Configuration
# %% Cell 8: Experiment Configuration (UPDATE)
@dataclass
class ExperimentConfig:
    """Configuration for a single experiment."""
    K: int = 10
    d0: int = 1000
    n_per_class: int = 10
    d_b: int = 64
    L_hidden: int = 5
    width_pattern: str = 'constant_narrow'
    wd: float = 0.0
    lr: float = 0.1
    max_iters: int = 800
    batch_size: int = -1
    seed: int = 42
    use_relu: bool = True
    # CONTROL EXPERIMENT ADDITION: New parameters
    optimizer_type: str = 'sgd'      # 'sgd', 'adam'
    init_type: str = 'kaiming'       # 'kaiming' (He), 'xavier'
    
    def to_hash(self) -> str:
        """Generate unique hash for config (excluding seed)."""
        config_dict = asdict(self)
        config_dict.pop('seed')
        config_str = str(sorted(config_dict.items()))
        return hashlib.md5(config_str.encode()).hexdigest()[:8]

# %% Cell 9: Lightning Module
# %% Cell 9: Lightning Module (UPDATE)
class BottleneckLightningModule(pl.LightningModule):
    def __init__(self, config: ExperimentConfig):
        super().__init__()
        self.save_hyperparameters()
        self.config = config
        
        self.model = BottleneckMLP(
            d0=config.d0,
            K=config.K,
            L_hidden=config.L_hidden,
            d_b=config.d_b,
            width_pattern=config.width_pattern,
            use_relu=config.use_relu,
            init_type=config.init_type # CONTROL EXPERIMENT ADDITION
        )
        self.criterion = nn.CrossEntropyLoss()
        
    # ... (forward, training_step, validation_step remain unchanged)
    def forward(self, x): return self.model(x)
    def training_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss
    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        # CONTROL EXPERIMENT ADDITION: Optimizer selection
        if self.config.optimizer_type.lower() == 'adam':
            optimizer = torch.optim.Adam(
                self.parameters(),
                lr=self.config.lr,
                weight_decay=self.config.wd
            )
        else:
            # Default to SGD
            optimizer = torch.optim.SGD(
                self.parameters(),
                lr=self.config.lr,
                momentum=0.9,
                weight_decay=self.config.wd
            )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.config.max_iters
        )
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}



# %% Cell 9b: Control Experiment Callbacks (NEW)
class RankTrackingCallback(pl.Callback):
    """CONTROL EXPERIMENT ADDITION: Track rank evolution during early training."""
    def __init__(self, check_interval_epochs=5):
        self.history = []
        self.check_interval = check_interval_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        # Only check every few epochs to save time
        if trainer.current_epoch % self.check_interval != 0:
            return
            
        with torch.no_grad():
            # Quick check on a small batch of data if available, else skip
            # For simplicity in this codebase, we use the validation set
            if trainer.val_dataloaders is None:
                return
                
            # Grab a batch
            try:
                batch = next(iter(trainer.val_dataloaders))
                X, _ = batch
                X = X.to(pl_module.device)
                
                activations = pl_module.model.get_layer_activations(X)
                
                # Compute rank of the middle layer (bottleneck or near it)
                L = len(activations) - 1
                mid_layer = L // 2 + 1
                if mid_layer < len(activations):
                    H = activations[mid_layer]
                    rank = MetricsComputer.stable_rank(H)
                    self.history.append({
                        'epoch': trainer.current_epoch,
                        'mid_layer_rank': rank
                    })
            except Exception:
                pass

def train_single_run(config: ExperimentConfig, timer: ExperimentTimer) -> Optional[Dict]:
    # ... (Start of existing function)
    if timer.should_stop():
        print("⏱️  Time limit reached, skipping this run")
        return None
    
    set_seed(config.seed)
    
    try:
        N = config.K * config.n_per_class
        batch_size = N if config.batch_size == -1 else config.batch_size
        train_loader, val_loader, dataset = create_data_loaders(
            config.K, config.d0, config.n_per_class, batch_size, config.seed
        )
        
        model = BottleneckLightningModule(config)
        
        early_stop = EarlyStopping(monitor='val_loss', patience=50, mode='min')
        
        # CONTROL EXPERIMENT ADDITION: Add Rank Callback
        rank_callback = RankTrackingCallback(check_interval_epochs=10)
        
        trainer = pl.Trainer(
            max_steps=config.max_iters,
            logger=False,
            callbacks=[early_stop, rank_callback], # Added callback here
            enable_progress_bar=False,
            enable_checkpointing=False,
            accelerator='auto',
            devices=1,
            deterministic=True
        )
        
        trainer.fit(model, train_loader, val_loader)
        
        # ... (Metrics computation logic remains the same)
        device = next(model.parameters()).device
        model.eval()
        metrics = MetricsComputer.compute_all_metrics(
            model.model, train_loader, config.K, str(device)
        )
        
        result = {
            'config': asdict(config),
            'metrics': metrics,
            'final_train_acc': trainer.callback_metrics.get('train_acc', torch.tensor(0.0)).item(),
            'final_val_loss': trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))).item(),
            # CONTROL EXPERIMENT ADDITION: Save rank history
            'rank_history': rank_callback.history 
        }
        
        del model, trainer, train_loader, val_loader, dataset
        cleanup_memory()
        
        return result
        
    except Exception as e:
        print(f"❌ Run failed: {e}")
        cleanup_memory()
        return None

# %% Cell 10: Training Function
def train_single_run(config: ExperimentConfig, timer: ExperimentTimer) -> Optional[Dict]:
    """Train a single configuration and return metrics."""
    if timer.should_stop():
        print("⏱️  Time limit reached, skipping this run")
        return None
    
    set_seed(config.seed)
    
    try:
        N = config.K * config.n_per_class
        batch_size = N if config.batch_size == -1 else config.batch_size
        train_loader, val_loader, dataset = create_data_loaders(
            config.K, config.d0, config.n_per_class, batch_size, config.seed
        )
        
        model = BottleneckLightningModule(config)
        
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=50,
            mode='min'
        )
        
        trainer = pl.Trainer(
            max_steps=config.max_iters,
            logger=False,
            callbacks=[early_stop],
            enable_progress_bar=False,
            enable_checkpointing=False,
            accelerator='auto',
            devices=1,
            deterministic=True
        )
        
        trainer.fit(model, train_loader, val_loader)
        
        device = next(model.parameters()).device
        model.eval()
        metrics = MetricsComputer.compute_all_metrics(
            model.model, train_loader, config.K, str(device)
        )
        
        result = {
            'config': asdict(config),
            'metrics': metrics,
            'final_train_acc': trainer.callback_metrics.get('train_acc', torch.tensor(0.0)).item(),
            'final_val_loss': trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))).item()
        }
        
        del model, trainer, train_loader, val_loader, dataset
        cleanup_memory()
        
        return result
        
    except Exception as e:
        print(f"❌ Run failed: {e}")
        cleanup_memory()
        return None

# %% Cell 11: Checkpoint Management
def save_checkpoint(results: List[Dict], checkpoint_path: Path):
    """Save intermediate results."""
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"💾 Checkpoint saved: {len(results)} results")

def load_checkpoint(checkpoint_path: Path) -> List[Dict]:
    """Load checkpoint if exists."""
    if checkpoint_path.exists():
        with open(checkpoint_path, 'rb') as f:
            results = pickle.load(f)
        print(f"📂 Loaded checkpoint: {len(results)} results")
        return results
    return []

# %% Cell 12: Sweep Function
def run_sweep(sweep_params: Dict, n_seeds: int, timer: ExperimentTimer, 
              checkpoint_path: Path) -> List[Dict]:
    """Run full hyperparameter sweep with time management."""
    
    all_results = load_checkpoint(checkpoint_path)
    
    keys = list(sweep_params.keys())
    values = [sweep_params[k] for k in keys]
    
    completed_configs = set()
    for res in all_results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'], 
               cfg['width_pattern'], cfg['wd'], cfg['seed'])
        completed_configs.add(key)
    
    config_count = 0
    total_configs = np.prod([len(v) for v in values]) * n_seeds
    
    for combo in product(*values):
        config_dict = dict(zip(keys, combo))
        
        for seed in range(n_seeds):
            config = ExperimentConfig(**config_dict, seed=seed)
            
            key = (config.n_per_class, config.d_b, config.L_hidden,
                   config.width_pattern, config.wd, seed)
            if key in completed_configs:
                continue
            
            config_count += 1
            
            if timer.should_stop():
                print(f"\n⏱️  Time limit reached after {config_count} runs")
                print(f"📊 Completed: {len(all_results)}/{total_configs} runs")
                break
            
            print(f"\n[{len(all_results)+1}/{total_configs}] n={config.n_per_class}, "
                  f"d_b={config.d_b}, L={config.L_hidden}, "
                  f"pattern={config.width_pattern[:10]}, wd={config.wd}, seed={seed}")
            timer.print_status()
            print(f"🔧 {get_memory_stats()}")
            
            try:
                result = train_single_run(config, timer)
                if result is not None:
                    all_results.append(result)
                    
                    if len(all_results) % 5 == 0:
                        save_checkpoint(all_results, checkpoint_path)
                        
            except Exception as e:
                print(f"❌ Run failed: {e}")
                cleanup_memory()
                continue
        
        if timer.should_stop():
            break
    
    save_checkpoint(all_results, checkpoint_path)
    
    return all_results

# %% Cell 13: ALL 11 VISUALIZATION FUNCTIONS WITH NaN/Inf HANDLING

def plot_rank_evolution(results: List[Dict], save_path: Path):
    """Plot stable rank evolution across layers."""
    if len(results) == 0:
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['width_pattern'])
        if key not in config_groups:
            config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, pattern = key
        
        layer_ranks = {}
        for res in group:
            metrics = res['metrics']
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    rank = layer_metrics['stable_rank']
                    if np.isfinite(rank):
                        if layer_idx not in layer_ranks:
                            layer_ranks[layer_idx] = []
                        layer_ranks[layer_idx].append(rank)
        
        if len(layer_ranks) == 0:
            continue
        
        layers = sorted(layer_ranks.keys())
        means = [np.mean(layer_ranks[l]) for l in layers]
        stds = [np.std(layer_ranks[l]) for l in layers]
        
        valid_indices = [i for i, (m, s) in enumerate(zip(means, stds)) 
                        if np.isfinite(m) and np.isfinite(s)]
        
        if len(valid_indices) == 0:
            continue
        
        layers_valid = [layers[i] for i in valid_indices]
        means_valid = [means[i] for i in valid_indices]
        stds_valid = [stds[i] for i in valid_indices]
        
        ax.plot(layers_valid, means_valid, 'o-', linewidth=2, markersize=8)
        ax.fill_between(layers_valid, 
                        np.array(means_valid) - np.array(stds_valid),
                        np.array(means_valid) + np.array(stds_valid),
                        alpha=0.3)
        
        ax.set_xlabel('Layer ℓ', fontsize=12)
        ax.set_ylabel('Stable Rank', fontsize=12)
        ax.set_title(f'n={n}, d_b={d_b}, {pattern}', fontsize=11)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_singular_spectra(results: List[Dict], save_path: Path):
    """Plot singular value spectra."""
    if len(results) == 0:
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups:
            config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, L = key
        
        for res in group[:1]:
            metrics = res['metrics']
            middle_layer = L // 2 + 1
            layer_key = f'layer_{middle_layer}'
            
            if layer_key in metrics:
                svals = metrics[layer_key]['singular_values']
                if np.isfinite(svals).all() and len(svals) > 0:
                    ax.loglog(range(1, len(svals)+1), svals, 'o-', linewidth=2)
        
        ax.set_xlabel('Singular Value Index', fontsize=12)
        ax.set_ylabel('Singular Value', fontsize=12)
        ax.set_title(f'n={n}, d_b={d_b}, L={L}', fontsize=11)
        ax.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_bottleneck_location(results: List[Dict], save_path: Path):
    """Histogram of bottleneck layer location."""
    if len(results) == 0:
        return
    
    bottleneck_layers = []
    
    for res in results:
        metrics = res['metrics']
        ranks = {}
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                layer_idx = int(layer_name.split('_')[1])
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    ranks[layer_idx] = rank
        
        if len(ranks) > 1:
            min_rank_layer = min(ranks, key=ranks.get)
            bottleneck_layers.append(min_rank_layer)
    
    if len(bottleneck_layers) == 0:
        return
    
    plt.figure(figsize=(10, 6))
    plt.hist(bottleneck_layers, bins=max(10, len(set(bottleneck_layers))), 
             edgecolor='black', alpha=0.7, color='steelblue')
    plt.xlabel('Layer ℓ*', fontsize=14)
    plt.ylabel('Frequency', fontsize=14)
    plt.title('Distribution of Bottleneck Layer Location', fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")
    print(f"  Mean bottleneck layer: {np.mean(bottleneck_layers):.2f}")
    print(f"  Median bottleneck layer: {np.median(bottleneck_layers):.2f}")

def plot_scaling_law(results: List[Dict], save_path: Path):
    """Plot scaling law analysis."""
    if len(results) < 10:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    min_rank = min(min_rank, rank)
        
        if min_rank < float('inf') and np.isfinite(min_rank):
            data.append({
                'n': cfg['n_per_class'],
                'K': cfg['K'],
                'd_b': cfg['d_b'],
                'nK': cfg['n_per_class'] * cfg['K'],
                'r_min': min_rank
            })
    
    if len(data) == 0:
        return
    
    df = pd.DataFrame(data)
    df = df[np.isfinite(df['r_min']) & np.isfinite(df['nK']) & np.isfinite(df['d_b'])]
    
    if len(df) == 0:
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].loglog(df['nK'], df['r_min'], 'o', alpha=0.5)
    axes[0].set_xlabel('nK (total samples)', fontsize=12)
    axes[0].set_ylabel('Min Stable Rank', fontsize=12)
    axes[0].set_title('Scaling with Data', fontsize=13)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].loglog(df['d_b'], df['r_min'], 'o', alpha=0.5)
    axes[1].set_xlabel('Bottleneck Width d_b', fontsize=12)
    axes[1].set_ylabel('Min Stable Rank', fontsize=12)
    axes[1].set_title('Scaling with Width', fontsize=13)
    axes[1].grid(True, alpha=0.3)
    
    axes[2].hist(df['r_min'], bins=20, alpha=0.7, edgecolor='black')
    axes[2].set_xlabel('Min Stable Rank', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].set_title('Distribution', fontsize=13)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_phase_diagram(results: List[Dict], save_path: Path):
    """Heatmap of normalized bottleneck rank."""
    if len(results) < 10:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    min_rank = min(min_rank, rank)
        
        if min_rank < float('inf') and np.isfinite(min_rank):
            nK = cfg['n_per_class'] * cfg['K']
            normalized_rank = min_rank / min(cfg['d_b'], nK)
            
            if np.isfinite(normalized_rank):
                data.append({
                    'log_n': np.log10(cfg['n_per_class']),
                    'log_d_b': np.log10(cfg['d_b']),
                    'norm_rank': normalized_rank
                })
    
    if len(data) == 0:
        return
    
    df = pd.DataFrame(data)
    df = df[np.isfinite(df['log_n']) & np.isfinite(df['log_d_b']) & np.isfinite(df['norm_rank'])]
    
    if len(df) == 0:
        return
    
    pivot = df.pivot_table(values='norm_rank', 
                           index='log_d_b', 
                           columns='log_n', 
                           aggfunc='mean')
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='viridis', 
                cbar_kws={'label': 'r_b / min(d_b, nK)'})
    plt.xlabel('log₁₀(n)', fontsize=14)
    plt.ylabel('log₁₀(d_b)', fontsize=14)
    plt.title('Phase Diagram: Normalized Bottleneck Rank', fontsize=16)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_training_dynamics(results: List[Dict], save_path: Path):
    """Plot rank and NC1 evolution across layers."""
    if len(results) == 0:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'])
        if key not in config_groups:
            config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:4]):
        ax = axes[idx]
        n, d_b = key
        
        layer_data = {}
        for res in group:
            metrics = res['metrics']
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    rank = layer_metrics['stable_rank']
                    nc1 = layer_metrics['nc1']
                    if np.isfinite(rank) and np.isfinite(nc1):
                        if layer_idx not in layer_data:
                            layer_data[layer_idx] = {'ranks': [], 'nc1': []}
                        layer_data[layer_idx]['ranks'].append(rank)
                        layer_data[layer_idx]['nc1'].append(nc1)
        
        if len(layer_data) == 0:
            continue
        
        layers = sorted(layer_data.keys())
        rank_means = [np.mean(layer_data[l]['ranks']) for l in layers]
        nc1_means = [np.mean(layer_data[l]['nc1']) for l in layers]
        
        ax2 = ax.twinx()
        ax.plot(layers, rank_means, 'b-o', label='Stable Rank', linewidth=2)
        ax2.plot(layers, nc1_means, 'r-s', label='NC1', linewidth=2)
        
        ax.set_xlabel('Layer', fontsize=12)
        ax.set_ylabel('Stable Rank', color='b', fontsize=12)
        ax2.set_ylabel('NC1', color='r', fontsize=12)
        ax.set_title(f'n={n}, d_b={d_b}', fontsize=13)
        ax.grid(True, alpha=0.3)
        
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='best')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_wd_ablation(results: List[Dict], save_path: Path):
    """Compare rank profiles with/without weight decay."""
    if len(results) < 10:
        return
    
    wd_values = sorted(list(set([r['config']['wd'] for r in results])))
    n_wd = len(wd_values)
    
    if n_wd == 0:
        return
    
    fig, axes = plt.subplots(1, min(n_wd, 4), figsize=(6*min(n_wd, 4), 5))
    if n_wd == 1:
        axes = [axes]
    
    for idx, wd in enumerate(wd_values[:4]):
        ax = axes[idx]
        
        wd_results = [r for r in results if r['config']['wd'] == wd]
        
        n_groups = {}
        for res in wd_results:
            n = res['config']['n_per_class']
            if n not in n_groups:
                n_groups[n] = []
            n_groups[n].append(res)
        
        for n, group in n_groups.items():
            layer_ranks = {}
            for res in group:
                metrics = res['metrics']
                for layer_name, layer_metrics in metrics.items():
                    if layer_name.startswith('layer_'):
                        layer_idx = int(layer_name.split('_')[1])
                        rank = layer_metrics['stable_rank']
                        if np.isfinite(rank):
                            if layer_idx not in layer_ranks:
                                layer_ranks[layer_idx] = []
                            layer_ranks[layer_idx].append(rank)
            
            if len(layer_ranks) == 0:
                continue
            
            layers = sorted(layer_ranks.keys())
            means = [np.mean(layer_ranks[l]) for l in layers]
            ax.plot(layers, means, '-o', label=f'n={n}', linewidth=2, markersize=6)
        
        ax.set_xlabel('Layer', fontsize=12)
        ax.set_ylabel('Stable Rank', fontsize=12)
        ax.set_title(f'Weight Decay = {wd}', fontsize=13)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_depth_analysis(results: List[Dict], save_path: Path):
    """Analyze effect of network depth on bottleneck formation."""
    if len(results) < 10:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        bottleneck_layer = None
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank) and rank < min_rank:
                    min_rank = rank
                    bottleneck_layer = int(layer_name.split('_')[1])
        
        if min_rank < float('inf') and np.isfinite(min_rank) and bottleneck_layer is not None:
            relative_depth = bottleneck_layer / (cfg['L_hidden'] + 2) if cfg['L_hidden'] > 0 else 0
            if np.isfinite(relative_depth):
                data.append({
                    'L': cfg['L_hidden'],
                    'd_b': cfg['d_b'],
                    'r_min': min_rank,
                    'bottleneck_layer': bottleneck_layer,
                    'relative_depth': relative_depth
                })
    
    if len(data) == 0:
        return
    
    df = pd.DataFrame(data)
    df = df[np.isfinite(df['r_min']) & np.isfinite(df['relative_depth'])]
    
    if len(df) == 0:
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for d_b in sorted(df['d_b'].unique()):
        subset = df[df['d_b'] == d_b]
        grouped = subset.groupby('L')['r_min'].agg(['mean', 'std']).reset_index()
        grouped = grouped[np.isfinite(grouped['mean']) & np.isfinite(grouped['std'])]
        if len(grouped) > 0:
            axes[0].errorbar(grouped['L'], grouped['mean'], yerr=grouped['std'], 
                            marker='o', label=f'd_b={d_b}', linewidth=2, capsize=5)
    
    axes[0].set_xlabel('Network Depth (L)', fontsize=12)
    axes[0].set_ylabel('Min Stable Rank', fontsize=12)
    axes[0].set_title('Rank vs Network Depth', fontsize=13)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].scatter(df['L'], df['bottleneck_layer'], alpha=0.5)
    if len(df) > 0:
        axes[1].plot([df['L'].min(), df['L'].max()], 
                     [df['L'].min()/2, df['L'].max()/2], 
                     'r--', linewidth=2, label='Middle layer')
    axes[1].set_xlabel('Network Depth (L)', fontsize=12)
    axes[1].set_ylabel('Bottleneck Layer', fontsize=12)
    axes[1].set_title('Bottleneck Location vs Depth', fontsize=13)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].hist(df['relative_depth'], bins=20, alpha=0.7, edgecolor='black')
    axes[2].set_xlabel('Relative Bottleneck Position', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].set_title('Normalized Bottleneck Location', fontsize=13)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_capacity_analysis(results: List[Dict], save_path: Path):
    """Analyze capacity constraints: r_b vs min(d_b, nK/2)."""
    if len(results) < 10:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    min_rank = min(min_rank, rank)
        
        if min_rank < float('inf') and np.isfinite(min_rank):
            nK = cfg['n_per_class'] * cfg['K']
            capacity_bound = min(cfg['d_b'], nK / 2)
            ratio = min_rank / capacity_bound if capacity_bound > 0 else 0
            
            if np.isfinite(ratio) and np.isfinite(capacity_bound):
                data.append({
                    'n': cfg['n_per_class'],
                    'd_b': cfg['d_b'],
                    'nK': nK,
                    'r_min': min_rank,
                    'capacity_bound': capacity_bound,
                    'ratio': ratio
                })
    
    if len(data) == 0:
        return
    
    df = pd.DataFrame(data)
    df = df[np.isfinite(df['r_min']) & np.isfinite(df['capacity_bound']) & 
            np.isfinite(df['ratio']) & (df['capacity_bound'] > 0)]
    
    if len(df) == 0:
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].scatter(df['capacity_bound'], df['r_min'], alpha=0.5, s=50)
    max_val = max(df['capacity_bound'].max(), df['r_min'].max())
    axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='y=x')
    axes[0].set_xlabel('Capacity Bound: min(d_b, nK/2)', fontsize=12)
    axes[0].set_ylabel('Actual Min Rank', fontsize=12)
    axes[0].set_title('Capacity Constraint Analysis', fontsize=13)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(df['ratio'], bins=30, alpha=0.7, edgecolor='black')
    axes[1].axvline(x=1.0, color='r', linestyle='--', linewidth=2, label='Perfect match')
    axes[1].set_xlabel('r_b / min(d_b, nK/2)', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Normalized Rank Distribution', fontsize=13)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    pivot = df.pivot_table(values='ratio', index='d_b', columns='n', aggfunc='mean')
    if not pivot.empty:
        im = axes[2].imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=2)
        axes[2].set_xticks(range(len(pivot.columns)))
        axes[2].set_yticks(range(len(pivot.index)))
        axes[2].set_xticklabels(pivot.columns)
        axes[2].set_yticklabels(pivot.index)
        axes[2].set_xlabel('n (samples per class)', fontsize=12)
        axes[2].set_ylabel('d_b (bottleneck width)', fontsize=12)
        axes[2].set_title('Capacity Utilization', fontsize=13)
        plt.colorbar(im, ax=axes[2], label='r_b / min(d_b, nK/2)')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")
    print(f"  Mean ratio: {df['ratio'].mean():.3f} ± {df['ratio'].std():.3f}")
    print(f"  Median ratio: {df['ratio'].median():.3f}")

def plot_accuracy_vs_rank(results: List[Dict], save_path: Path):
    """Analyze relationship between bottleneck rank and accuracy."""
    if len(results) < 10:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    min_rank = min(min_rank, rank)
        
        if min_rank < float('inf') and np.isfinite(min_rank):
            acc = res.get('final_train_acc', 0)
            if np.isfinite(acc):
                data.append({
                    'r_min': min_rank,
                    'acc': acc,
                    'd_b': cfg['d_b'],
                    'n': cfg['n_per_class']
                })
    
    if len(data) == 0:
        return
    
    df = pd.DataFrame(data)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    scatter = axes[0].scatter(df['r_min'], df['acc'], c=df['d_b'], 
                             cmap='viridis', s=50, alpha=0.6)
    axes[0].set_xlabel('Min Stable Rank', fontsize=12)
    axes[0].set_ylabel('Training Accuracy', fontsize=12)
    axes[0].set_title('Accuracy vs Bottleneck Rank', fontsize=13)
    axes[0].grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axes[0], label='d_b')
    
    df['rank_bin'] = pd.cut(df['r_min'], bins=5)
    df.boxplot(column='acc', by='rank_bin', ax=axes[1])
    axes[1].set_xlabel('Rank Bin', fontsize=12)
    axes[1].set_ylabel('Training Accuracy', fontsize=12)
    axes[1].set_title('Accuracy Distribution by Rank', fontsize=13)
    axes[1].get_figure().suptitle('')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_architecture_comparison(results: List[Dict], save_path: Path):
    """Compare different architecture patterns."""
    if len(results) < 10:
        return
    
    patterns = list(set([r['config']['width_pattern'] for r in results]))
    if len(patterns) < 2:
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    min_rank = min(min_rank, rank)
        
        if min_rank < float('inf'):
            data.append({
                'pattern': cfg['width_pattern'],
                'r_min': min_rank,
                'd_b': cfg['d_b']
            })
    
    df = pd.DataFrame(data)
    
    df.boxplot(column='r_min', by='pattern', ax=axes[0])
    axes[0].set_xlabel('Architecture Pattern', fontsize=12)
    axes[0].set_ylabel('Min Stable Rank', fontsize=12)
    axes[0].set_title('Rank by Architecture', fontsize=13)
    axes[0].get_figure().suptitle('')
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    sns.violinplot(data=df, x='pattern', y='r_min', hue='d_b', ax=axes[1], split=False)
    axes[1].set_xlabel('Architecture Pattern', fontsize=12)
    axes[1].set_ylabel('Min Stable Rank', fontsize=12)
    axes[1].set_title('Rank Distribution by Pattern and Width', fontsize=13)
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def generate_summary_table(results: List[Dict], save_path: Path):
    """Generate summary statistics table."""
    if len(results) == 0:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        ranks = {}
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                layer_idx = int(layer_name.split('_')[1])
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank):
                    ranks[layer_idx] = rank
        
        if len(ranks) > 0:
            min_rank = min(ranks.values())
            data.append({
                'n': cfg['n_per_class'],
                'd_b': cfg['d_b'],
                'L': cfg['L_hidden'],
                'pattern': cfg['width_pattern'][:10],
                'wd': cfg['wd'],
                'r_min': min_rank,
                'acc': res['final_train_acc']
            })
    
    df = pd.DataFrame(data)
    summary = df.groupby(['n', 'd_b', 'L', 'pattern', 'wd']).agg({
        'r_min': ['mean', 'std'],
        'acc': 'mean'
    }).round(3)
    
    summary.to_csv(save_path)
    print(f"✓ Saved: {save_path.name}")
    print("\nSummary Statistics:")
    print(summary.head(10))

# %% Cell 13b: Control Experiment Plots (NEW)

def plot_robustness_rank_depth(results: List[Dict], save_path: Path):
    """CONTROL EXPERIMENT ADDITION: Rank vs Depth grouped by Optimizer/Init."""
    if len(results) == 0: return
    
    # Filter for valid control results
    df_rows = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        for layer_name, m in metrics.items():
            if layer_name.startswith('layer_'):
                layer_idx = int(layer_name.split('_')[1])
                df_rows.append({
                    'layer': layer_idx,
                    'rank': m['stable_rank'],
                    'optimizer': cfg.get('optimizer_type', 'sgd'),
                    'init': cfg.get('init_type', 'kaiming'),
                    'lr': cfg.get('lr', 0.1),
                    'n': cfg['n_per_class']
                })
    
    df = pd.DataFrame(df_rows)
    if df.empty: return

    # Plot 1: Optimizer Comparison
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df, x='layer', y='rank', hue='optimizer', style='init', markers=True)
    plt.title('Robustness: Rank Profile by Optimizer & Init')
    plt.ylabel('Stable Rank')
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path / 'robustness_rank_optimizer.png', bbox_inches='tight')
    plt.close()

def plot_early_training_evolution(results: List[Dict], save_path: Path):
    """CONTROL EXPERIMENT ADDITION: Rank evolution over epochs."""
    history_rows = []
    for res in results:
        cfg = res['config']
        if 'rank_history' in res:
            for entry in res['rank_history']:
                history_rows.append({
                    'epoch': entry['epoch'],
                    'rank': entry['mid_layer_rank'],
                    'optimizer': cfg.get('optimizer_type', 'sgd'),
                    'lr': cfg.get('lr', 0.1)
                })
    
    df = pd.DataFrame(history_rows)
    if df.empty: return

    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df, x='epoch', y='rank', hue='optimizer', style='lr')
    plt.title('Early Training Rank Evolution (Mid-Layer)')
    plt.ylabel('Stable Rank')
    plt.xlabel('Epoch')
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path / 'robustness_training_evolution.png', bbox_inches='tight')
    plt.close()

def plot_robustness_spectrum(results: List[Dict], save_path: Path):
    """CONTROL EXPERIMENT ADDITION: Spectrum decay across controls."""
    plt.figure(figsize=(12, 5))
    
    for i, res in enumerate(results[:20]): # Sample subset
        cfg = res['config']
        metrics = res['metrics']
        # Find middle layer
        layer_keys = [k for k in metrics.keys() if k.startswith('layer_')]
        if not layer_keys: continue
        mid_layer = layer_keys[len(layer_keys)//2]
        
        svals = metrics[mid_layer]['singular_values']
        if len(svals) > 0:
            color = 'blue' if cfg.get('optimizer_type') == 'sgd' else 'orange'
            style = '-' if cfg.get('init_type') == 'kaiming' else '--'
            plt.plot(svals, color=color, linestyle=style, alpha=0.5)

    # Dummy lines for legend
    from matplotlib.lines import Line2D
    custom_lines = [Line2D([0], [0], color='blue', lw=2),
                    Line2D([0], [0], color='orange', lw=2),
                    Line2D([0], [0], color='gray', linestyle='-'),
                    Line2D([0], [0], color='gray', linestyle='--')]
    
    plt.legend(custom_lines, ['SGD', 'Adam', 'Kaiming', 'Xavier'])
    plt.yscale('log')
    plt.title('Singular Value Spectrum Robustness')
    plt.ylabel('Singular Value (Log Scale)')
    plt.xlabel('Index')
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path / 'robustness_spectrum.png', bbox_inches='tight')
    plt.close()

# %% Cell 14: Quick Test Run
print("\n" + "="*80)
print("🧪 RUNNING QUICK TEST (should complete in <1 minute)")
print("="*80)

test_timer = ExperimentTimer(max_hours=0.1)
test_config = ExperimentConfig(
    n_per_class=5,
    d_b=32,
    L_hidden=3,
    max_iters=100,
    width_pattern='constant_narrow',
    wd=0.0,
    seed=42
)

print(f"Testing with: n=5, d_b=32, L=3, max_iters=100")
test_result = train_single_run(test_config, test_timer)

if test_result is not None:
    print("✅ TEST PASSED! Proceeding to full sweep...")
    cleanup_memory()
else:
    print("❌ TEST FAILED! Check configuration and try again.")
    sys.exit(1)

# %% Cell 15: Full Sweep Configuration
print("\n" + "="*80)
print("🚀 FULL SWEEP CONFIGURATION")
print("="*80)

# Base LR reference
BASE_LR = 0.1

sweep_params = {
    'n_per_class': [1, 2, 5, 25,  40 , 60 , 100 , 250],
    'd_b': [1,2, 8,   64,  256 , 32 , 128 , 512,1024],
    'L_hidden': [ 5, 15, 20 , 30 , 40 , 50 , 100],
    'width_pattern': ['constant_narrow'],
    'wd': [ 1e-3],
    'K': [10],
        # CONTROL EXPERIMENT ADDITION: Robustness params
    'optimizer_type': ['sgd', 'adam'],
    'init_type': ['kaiming', 'xavier'],
    'lr': [BASE_LR, BASE_LR * 0.3]
}

n_seeds = 1
max_iters = 800

total_configs = 1
for param_name, param_values in sweep_params.items():
    total_configs *= len(param_values)

total_runs = total_configs * n_seeds
estimated_time_per_run = 3.46 / 60

runs_in_10h = int(10 * 60 / estimated_time_per_run)

print(f"Unique configurations: {total_configs}")
print(f"Seeds per config: {n_seeds}")
print(f"Total runs planned: {total_runs}")
print(f"Runs achievable in 10h: ~{runs_in_10h}")
print(f"Expected completion: ~{total_runs * estimated_time_per_run / 60:.1f} hours")
print(f"Max iterations per run: {max_iters}")
print("="*80)

print("\nParameter space breakdown:")
for param, values in sweep_params.items():
    print(f"  {param}: {values}")
    print(f"    → {len(values)} values")

print("\n📊 Expected coverage:")
print(f"  Sample sizes: {len(sweep_params['n_per_class'])} values (5 to 50)")
print(f"  Bottleneck widths: {len(sweep_params['d_b'])} values (8 to 256)")
print(f"  Network depths: {len(sweep_params['L_hidden'])} values (3 to 20)")
print(f"  Weight decay: {len(sweep_params['wd'])} values (0 to 0.01)")
print(f"  Architecture patterns: {len(sweep_params['width_pattern'])} types")

print("\n🎯 This configuration will:")
print("  ✓ Cover broader range of data scarcity (n=5 to n=50)")
print("  ✓ Test wider bottlenecks (8 to 256)")
print("  ✓ Explore deeper networks (up to 20 hidden layers)")
print("  ✓ More regularization strengths (6 wd values)")
print("  ✓ Generate publishable results with good coverage")

print("="*80)

ExperimentConfig.__dataclass_fields__['max_iters'].default = max_iters

# %% Cell 16: Run Full Sweep
print("\n🏃 Starting full sweep...")
main_timer = ExperimentTimer(max_hours=11.5)
checkpoint_path = CHECKPOINT_DIR / 'sweep_checkpoint.pkl'

all_results = run_sweep(
    sweep_params=sweep_params,
    n_seeds=n_seeds,
    timer=main_timer,
    checkpoint_path=checkpoint_path
)

print(f"\n✅ Sweep complete! {len(all_results)} results collected.")
print(f"⏱️  Total time: {main_timer.elapsed_hours():.2f} hours")

# %% Cell 17: Generate All Plots
print("\n" + "="*80)
print("📊 GENERATING VISUALIZATIONS")
print("="*80)

if len(all_results) > 0:
    print(f"Processing {len(all_results)} results...")
    
    plot_rank_evolution(all_results, PLOTS_DIR / 'rank_evolution.png')
    plot_singular_spectra(all_results, PLOTS_DIR / 'singular_spectra.png')
    plot_bottleneck_location(all_results, PLOTS_DIR / 'bottleneck_location.png')
    plot_scaling_law(all_results, PLOTS_DIR / 'scaling_law.png')
    plot_phase_diagram(all_results, PLOTS_DIR / 'phase_diagram.png')
    plot_training_dynamics(all_results, PLOTS_DIR / 'training_dynamics.png')
    plot_wd_ablation(all_results, PLOTS_DIR / 'wd_ablation.png')
    plot_depth_analysis(all_results, PLOTS_DIR / 'depth_analysis.png')
    plot_capacity_analysis(all_results, PLOTS_DIR / 'capacity_analysis.png')
    plot_accuracy_vs_rank(all_results, PLOTS_DIR / 'accuracy_vs_rank.png')
    plot_architecture_comparison(all_results, PLOTS_DIR / 'architecture_comparison.png')
    generate_summary_table(all_results, RESULTS_DIR / 'summary_stats.csv')
     # CONTROL EXPERIMENT ADDITION: Generate robustness plots
    print("📊 Generating Control Experiment Plots...")
    plot_robustness_rank_depth(all_results, PLOTS_DIR)
    plot_early_training_evolution(all_results, PLOTS_DIR)
    plot_robustness_spectrum(all_results, PLOTS_DIR)
    
    # Save final results
    with open(RESULTS_DIR / 'all_results.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    print(f"\n✓ Saved all results: {RESULTS_DIR / 'all_results.pkl'}")
    
    # Export to CSV
    rows = []
    for res in all_results:
        cfg = res['config']
        base_row = {
            'n_per_class': cfg['n_per_class'],
            'd_b': cfg['d_b'],
            'L_hidden': cfg['L_hidden'],
            'width_pattern': cfg['width_pattern'],
            'wd': cfg['wd'],
            'seed': cfg['seed'],
            'final_train_acc': res.get('final_train_acc', 0),
            'final_val_loss': res.get('final_val_loss', float('inf'))
        }
        
        metrics = res['metrics']
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                row = base_row.copy()
                row['layer'] = layer_name
                row['stable_rank'] = layer_metrics['stable_rank']
                row['nc1'] = layer_metrics['nc1']
                row['width'] = layer_metrics['width']
                rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / 'detailed_results.csv', index=False)
    print(f"✓ Saved detailed CSV: {RESULTS_DIR / 'detailed_results.csv'}")
    
else:
    print("⚠️  No results to visualize")

# %% Cell 18: Final Summary
print("\n" + "="*80)
print("📈 EXPERIMENT SUMMARY")
print("="*80)

if len(all_results) > 0:
    print(f"Total successful runs: {len(all_results)}")
    print(f"Total configurations tested: {len(set([(r['config']['n_per_class'], r['config']['d_b'], r['config']['L_hidden']) for r in all_results]))}")
    
    # Calculate key statistics
    min_ranks = []
    final_accs = []
    
    for res in all_results:
        metrics = res['metrics']
        ranks = [m['stable_rank'] for m in metrics.values() if 'stable_rank' in m]
        if ranks:
            min_ranks.append(min(ranks))
        final_accs.append(res.get('final_train_acc', 0))
    
    if min_ranks:
        print(f"\nBottleneck Characteristics:")
        print(f"  Mean minimum rank: {np.mean(min_ranks):.2f} ± {np.std(min_ranks):.2f}")
        print(f"  Median minimum rank: {np.median(min_ranks):.2f}")
    
    if final_accs:
        print(f"\nTraining Performance:")
        print(f"  Mean final accuracy: {np.mean(final_accs):.3f}")
        print(f"  Runs with acc > 0.9: {sum(np.array(final_accs) > 0.9)}/{len(final_accs)}")
    
    print(f"\n📁 Output Files:")
    print(f"  Results: {RESULTS_DIR}/")
    print(f"  Plots: {PLOTS_DIR}/")
    print(f"  Checkpoint: {checkpoint_path}")
    
else:
    print("⚠️  No results generated")

print("="*80)
print("\n✅ EXPERIMENT COMPLETE!")
print(f"⏱️  Total runtime: {main_timer.elapsed_hours():.2f} hours")
print("="*80)

Seed set to 42
Seed set to 42


✓ Imports successful!
✓ Device: cuda
✓ GPU: Tesla P100-PCIE-16GB
✓ GPU Memory: 17.06 GB
✓ Results directory: /kaggle/working/results
✓ Plots directory: /kaggle/working/plots
✓ PyTorch: 2.8.0+cu126
✓ Lightning: 2.6.0

🧪 RUNNING QUICK TEST (should complete in <1 minute)
Testing with: n=5, d_b=32, L=3, max_iters=100


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BottleneckMLP    │ 34.5 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 34.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 34.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_steps=100` reached.
Seed set to 0
Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


... (17,503 outputs left out)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BottleneckMLP    │  104 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 104 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 104 M                                                                                                
Total estimated model params size (MB): 419                                                                        
Modules in train mode: 204                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

💾 Checkpoint saved: 4032 results

✅ Sweep complete! 4032 results collected.
⏱️  Total time: 6.75 hours

📊 GENERATING VISUALIZATIONS
Processing 4032 results...
✓ Saved: rank_evolution.png
✓ Saved: singular_spectra.png
✓ Saved: bottleneck_location.png
  Mean bottleneck layer: 21.00
  Median bottleneck layer: 9.00
✓ Saved: scaling_law.png
✓ Saved: phase_diagram.png
✓ Saved: training_dynamics.png
✓ Saved: wd_ablation.png
✓ Saved: depth_analysis.png
✓ Saved: capacity_analysis.png
  Mean ratio: 0.216 ± 0.315
  Median ratio: 0.089
✓ Saved: accuracy_vs_rank.png
✓ Saved: summary_stats.csv

Summary Statistics:
                           r_min         acc
                            mean  std   mean
n d_b L   pattern    wd                     
1 1   5   constant_n 0.001   1.0  0.0  0.181
      15  constant_n 0.001   1.0  0.0  0.111
      20  constant_n 0.001   1.0  0.0  0.111
      30  constant_n 0.001   1.0  0.0  0.111
      40  constant_n 0.001   1.0  0.0  0.111
      50  constant_n 0.001   1.0